In [3]:
import xarray as xr
import pandas as pd 

In [2]:
ds = xr.open_dataset('~/Downloads/noaa_crw_thermal_history_annual_history_v3.7.0.nc', chunks='auto')

print(ds)

print(dict(ds.dims))

print(list(ds.data_vars.keys()))

<xarray.Dataset> Size: 8GB
Dimensions:          (years: 41, lat: 1390, lon: 7200)
Coordinates:
  * years            (years) int32 164B 1985 1986 1987 1988 ... 2023 2024 2025
  * lat              (lat) float64 11kB -35.27 -35.22 -35.17 ... 34.12 34.18
  * lon              (lon) float64 58kB -180.0 -179.9 -179.9 ... 179.9 180.0
Data variables:
    reef_mask        (lat, lon) int8 10MB dask.array<chunksize=(1390, 7200), meta=np.ndarray>
    mask             (lat, lon) int8 10MB dask.array<chunksize=(1390, 7200), meta=np.ndarray>
    ann_max_sst      (years, lat, lon) float32 2GB dask.array<chunksize=(17, 616, 3202), meta=np.ndarray>
    ann_max_anom     (years, lat, lon) float32 2GB dask.array<chunksize=(17, 616, 3202), meta=np.ndarray>
    ann_max_dhw      (years, lat, lon) float32 2GB dask.array<chunksize=(17, 616, 3202), meta=np.ndarray>
    withinyear_mean  (years, lat, lon) float32 2GB dask.array<chunksize=(17, 616, 3202), meta=np.ndarray>
    withinyear_sd    (years, lat, lon) float

/var/folders/ws/dmbkn8cx75l33lp2zph5rpbw0000gn/T/ipykernel_64327/908729591.py:5: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(dict(ds.dims))


In [4]:
caribbean = ds['ann_max_dhw'].sel(
    lat=slice(17, 32),           # Caribbean latitude
    lon=slice(-87.5, -64),       # Caribbean longitude
    years=[2016, 2018, 2020]     
)

print(f"Extracted dataframe shape: {caribbean.shape}")
print(f"Years included: {caribbean.coords['years'].values}")

thermal_df = caribbean.to_dataframe().reset_index()
thermal_df.columns = ['latitude', 'longitude', 'year', 'annual_dhw']

print(f"\nDataframe shape: {thermal_df.shape}")
print(f"DHW range by year:")
print(thermal_df.groupby('year')['annual_dhw'].agg(['min', 'max', 'mean']))

thermal_df.to_csv('thermal_caribbean_2016_2018_2020.csv', index=False)
print(f"Saved: thermal_caribbean_2016_2018_2020.csv")

Extracted dataframe shape: (3, 300, 470)
Years included: [2016 2018 2020]

Dataframe shape: (423000, 4)
DHW range by year:
            min        max      mean
year                                
-87.475006  0.0  13.748572  4.250131
-87.425003  0.0  11.467146  3.855227
-87.375008  0.0   9.769995  3.645167
-87.325005  0.0   9.481423  3.431712
-87.275002  0.0   8.581423  3.361282
...         ...        ...       ...
-64.225006  0.0   3.854284  1.598994
-64.175003  0.0   3.705713  1.530884
-64.125008  0.0   3.248570  1.448571
-64.075005  0.0   3.122857  1.415595
-64.025002  0.0   3.112857  1.382857

[470 rows x 3 columns]
Saved: thermal_caribbean_2016_2018_2020.csv


In [5]:
print(thermal_df.head(10))

   latitude  longitude       year  annual_dhw
0      2016  17.025002 -87.475006         NaN
1      2016  17.025002 -87.425003         NaN
2      2016  17.025002 -87.375008         NaN
3      2016  17.025002 -87.325005         NaN
4      2016  17.025002 -87.275002         NaN
5      2016  17.025002 -87.225006         NaN
6      2016  17.025002 -87.175003         NaN
7      2016  17.025002 -87.125008         NaN
8      2016  17.025002 -87.075005         NaN
9      2016  17.025002 -87.025002         NaN


## Looking for NaN values

In [6]:
thermal_df = pd.read_csv('thermal_caribbean_2016_2018_2020.csv')

print(f"Total rows: {len(thermal_df):,}")
print(f"Total NaN values: {thermal_df['annual_dhw'].isnull().sum():,}")
print(f"Percentage NaN: {thermal_df['annual_dhw'].isnull().sum() / len(thermal_df) * 100:.1f}%")

print("NaN by year:")
for year in [2016, 2018, 2020]:
    year_data = thermal_df[thermal_df['year'] == year]
    nan_count = year_data['annual_dhw'].isnull().sum()
    print(f"  {year}: {nan_count:,} NaN out of {len(year_data):,} ({nan_count/len(year_data)*100:.1f}%)")

print(f"\nNon-NaN rows: {thermal_df['annual_dhw'].notna().sum():,}")

Total rows: 423,000
Total NaN values: 386,157
Percentage NaN: 91.3%
NaN by year:
  2016: 0 NaN out of 0 (nan%)
  2018: 0 NaN out of 0 (nan%)
  2020: 0 NaN out of 0 (nan%)

Non-NaN rows: 36,843


/var/folders/ws/dmbkn8cx75l33lp2zph5rpbw0000gn/T/ipykernel_64327/2071493421.py:11: RuntimeWarning: invalid value encountered in scalar divide
  print(f"  {year}: {nan_count:,} NaN out of {len(year_data):,} ({nan_count/len(year_data)*100:.1f}%)")


The warning is caused due to the presence of NaN values in the calculation. Since we will be getting rid of these values in the upcoming steps, we can ignore it for now. 

## STEP 1: LOADING DATA

In [7]:
thermal_raw = pd.read_csv('thermal_caribbean_2016_2018_2020.csv')

print(f"  Shape: {thermal_raw.shape}")
print(f"  Columns: {list(thermal_raw.columns)}")
print(f"  Data types:\n{thermal_raw.dtypes}")

print(thermal_raw.head())

  Shape: (423000, 4)
  Columns: ['latitude', 'longitude', 'year', 'annual_dhw']
  Data types:
latitude        int64
longitude     float64
year          float64
annual_dhw    float64
dtype: object
   latitude  longitude       year  annual_dhw
0      2016  17.025002 -87.475006         NaN
1      2016  17.025002 -87.425003         NaN
2      2016  17.025002 -87.375008         NaN
3      2016  17.025002 -87.325005         NaN
4      2016  17.025002 -87.275002         NaN


## STEP 2: ASSESSING TIDINESS

In the dataframe, each row stands for one grid point in that one year. Each column is one variable and the dataframe has four variables namely:
- year
- latitude
- longitude
- annual_dhw

There are no nested data structures and all values are in single cells. There is no need for restructuring the dataset. 
Hence, the data is tidy. 

## STEP 3: DATASET SIZE

In [20]:
print("Column names and first row:")
print(thermal_raw.columns.tolist())
print("\nFirst row:")
print(thermal_raw.iloc[0])
print("\nData types:")
print(thermal_raw.dtypes)

Column names and first row:
['year', 'latitude', 'longitude', 'annual_dhw']

First row:
year          2016.000000
latitude        17.025002
longitude      -87.475006
annual_dhw            NaN
Name: 0, dtype: float64

Data types:
year            int64
latitude      float64
longitude     float64
annual_dhw    float64
dtype: object


At this point, we noticed an issue of column renaming with the dataset. We will reload the dataset and fix the error. 

In [25]:
thermal_raw = pd.read_csv('thermal_caribbean_2016_2018_2020.csv')

# The columns are actually: year, latitude, longitude, annual_dhw
# But pandas read them as: latitude, longitude, year, annual_dhw
# So we need to swap the values back

# Create a new dataframe with correct mapping
thermal_raw_fixed = pd.DataFrame({
    'year': thermal_raw['latitude'].astype(int),
    'latitude': thermal_raw['longitude'],
    'longitude': thermal_raw['year'],
    'annual_dhw': thermal_raw['annual_dhw']
})

print("After fixing:")
print(thermal_raw_fixed.dtypes)
print("\nFirst few rows:")
print(thermal_raw_fixed.head(10))
print("\nYear range:", thermal_raw_fixed['year'].unique())
print("Latitude range:", thermal_raw_fixed['latitude'].min(), "to", thermal_raw_fixed['latitude'].max())

After fixing:
year            int64
latitude      float64
longitude     float64
annual_dhw    float64
dtype: object

First few rows:
   year   latitude  longitude  annual_dhw
0  2016  17.025002 -87.475006         NaN
1  2016  17.025002 -87.425003         NaN
2  2016  17.025002 -87.375008         NaN
3  2016  17.025002 -87.325005         NaN
4  2016  17.025002 -87.275002         NaN
5  2016  17.025002 -87.225006         NaN
6  2016  17.025002 -87.175003         NaN
7  2016  17.025002 -87.125008         NaN
8  2016  17.025002 -87.075005         NaN
9  2016  17.025002 -87.025002         NaN

Year range: [2016 2018 2020]
Latitude range: 17.025001525878906 to 31.975006103515625


In [26]:
thermal_raw = thermal_raw_fixed

In [27]:
print(f"  Total rows: {len(thermal_raw):,}")
print(f"  Total columns: {len(thermal_raw.columns)}")
print(f"  Total cells: {len(thermal_raw) * len(thermal_raw.columns):,}")
print(f"\nSpatial coverage:")
print(f"  Latitude range: {thermal_raw['latitude'].min():.4f}° to {thermal_raw['latitude'].max():.4f}°N")
print(f"  Longitude range: {thermal_raw['longitude'].min():.4f}° to {thermal_raw['longitude'].max():.4f}°W")
print(f"  Unique latitude points: {thermal_raw['latitude'].nunique():,}")
print(f"  Unique longitude points: {thermal_raw['longitude'].nunique():,}")
print(f"\nTemporal coverage:")
print(f"  Years: {sorted(thermal_raw['year'].unique())}")
print(f"  Rows per year: 141,000 each")
mem_mb = thermal_raw.memory_usage(deep=True).sum() / 1024 / 1024
print(f"\nMemory usage: {mem_mb:.2f} MB")

  Total rows: 423,000
  Total columns: 4
  Total cells: 1,692,000

Spatial coverage:
  Latitude range: 17.0250° to 31.9750°N
  Longitude range: -87.4750° to -64.0250°W
  Unique latitude points: 300
  Unique longitude points: 470

Temporal coverage:
  Years: [np.int64(2016), np.int64(2018), np.int64(2020)]
  Rows per year: 141,000 each

Memory usage: 12.91 MB


## STEP 4: MISSING DATA ANALYSIS

In [29]:
missing_total = thermal_raw.isnull().sum().sum()
print(f"\nTotal missing values: {missing_total:,}")

print(f"\nMissing by column:")
for col in thermal_raw.columns:
    missing_count = thermal_raw[col].isnull().sum()
    missing_pct = missing_count / len(thermal_raw) * 100
    print(f"  {col}: {missing_count:,} ({missing_pct:.1f}%)")

print(f"\nMissing by year (annual_dhw only):")
for year in sorted(thermal_raw['year'].unique()):
    year_data = thermal_raw[thermal_raw['year'] == year]
    missing = year_data['annual_dhw'].isnull().sum()
    total = len(year_data)
    pct = missing / total * 100
    print(f"  {int(year)}: {missing:,} / {total:,} ({pct:.1f}%)")


Total missing values: 386,157

Missing by column:
  year: 0 (0.0%)
  latitude: 0 (0.0%)
  longitude: 0 (0.0%)
  annual_dhw: 386,157 (91.3%)

Missing by year (annual_dhw only):
  2016: 128,719 / 141,000 (91.3%)
  2018: 128,719 / 141,000 (91.3%)
  2020: 128,719 / 141,000 (91.3%)


The unusually high value of missing values makes sense because the thermal data is a global 5km satellite grid, but we only extracted the Caribbean region values. Coral reefs, in general, cover a very small fraction of the ocean, estimated to be less than 1% of the ocean floor. So most of these grid points refer to ocean areas with no coral reefs. Only (100-91.3 =) 8.7% of the grid points in the dataset contain data from actual coral reef areas. Non reed areas show up as NaN values because there is no thermal stress to measure. 

## STEP 5: OUTLIER AND SUSPICIOUS DATA DETECTION

In [32]:
dhw = thermal_raw['annual_dhw'].dropna()

print(f"\nAnnual DHW statistics")
print(f"  Range: {dhw.min():.2f} to {dhw.max():.2f} °C-weeks")
print(f"  Mean: {dhw.mean():.2f} °C-weeks")
print(f"  Median: {dhw.median():.2f} °C-weeks")
print(f"  Std Dev: {dhw.std():.2f} °C-weeks")
print(f"  25th percentile: {dhw.quantile(0.25):.2f}")
print(f"  75th percentile: {dhw.quantile(0.75):.2f}")


Q1 = dhw.quantile(0.25)
Q3 = dhw.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = ((dhw < lower) | (dhw > upper)).sum()
print(f" Outliers (IQR method >1.5×IQR):")
print(f"  Count: {outliers:,} values")
print(f"  IQR bounds: {lower:.2f} to {upper:.2f}")

print(f" Validity checks:")
print(f"  All values non-negative: {(dhw >= 0).all()}")

lat_valid = (thermal_raw['latitude'] >= 17) & (thermal_raw['latitude'] <= 32)
lon_valid = (thermal_raw['longitude'] >= -87.5) & (thermal_raw['longitude'] <= -64)
print(f"\nGeographic validity:")
print(f"  Latitude 17-32°N: {lat_valid.sum():,} / {len(thermal_raw)} ")
print(f"  Longitude -87.5 to -64°W: {lon_valid.sum():,} / {len(thermal_raw)} ")


Annual DHW statistics
  Range: 0.00 to 16.44 °C-weeks
  Mean: 1.63 °C-weeks
  Median: 0.76 °C-weeks
  Std Dev: 2.06 °C-weeks
  25th percentile: 0.00
  75th percentile: 2.68
 Outliers (IQR method >1.5×IQR):
  Count: 1,122 values
  IQR bounds: -4.03 to 6.71
 Validity checks:
  All values non-negative: True

Geographic validity:
  Latitude 17-32°N: 423,000 / 423000 
  Longitude -87.5 to -64°W: 423,000 / 423000 


There seem to be no invalid data entries in the dataset. The outliers in temperature values can be attributed to the fact that 2016 was part of the third global bleaching event declared by the National Oceanic and Atmospheric Administration (NOAA) in 2015. 

## Data Cleaning 

In this step, we will be removing NaN values from the dataset. The small percentage of data (8.7%) that will remain after the removal of all NaN values should be sufficient for our carribean benthic sites. This will ensure that our data is clean in addition to being tidy. 

In [34]:
thermal_clean = thermal_raw.dropna(subset=['annual_dhw'])

print(f"  Original rows: {len(thermal_raw):,}")
print(f"  After removing NaN: {len(thermal_clean):,}")
print(f"  Rows removed: {len(thermal_raw) - len(thermal_clean):,}")

print(f"  Result verification:")
print(f"  Remaining NaN values: {thermal_clean['annual_dhw'].isnull().sum()}")
print(f"  Shape: {thermal_clean.shape}")

  Original rows: 423,000
  After removing NaN: 36,843
  Rows removed: 386,157
  Result verification:
  Remaining NaN values: 0
  Shape: (36843, 4)


## Summary statistics

In [35]:
print(f"\nAnnual DHW statistics by year (cleaned data):")
for year in sorted(thermal_clean['year'].unique()):
    year_data = thermal_clean[thermal_clean['year'] == year]['annual_dhw']
    print(f"\n  Year {int(year)}:")
    print(f"    Count: {len(year_data):,} grid points")
    print(f"    Mean: {year_data.mean():.2f} °C-weeks")
    print(f"    Median: {year_data.median():.2f} °C-weeks")
    print(f"    Range: {year_data.min():.2f} - {year_data.max():.2f} °C-weeks")
    print(f"    Std Dev: {year_data.std():.2f} °C-weeks")


Annual DHW statistics by year (cleaned data):

  Year 2016:
    Count: 12,281 grid points
    Mean: 1.54 °C-weeks
    Median: 0.82 °C-weeks
    Range: 0.00 - 12.21 °C-weeks
    Std Dev: 1.83 °C-weeks

  Year 2018:
    Count: 12,281 grid points
    Mean: 0.63 °C-weeks
    Median: 0.00 °C-weeks
    Range: 0.00 - 8.95 °C-weeks
    Std Dev: 1.15 °C-weeks

  Year 2020:
    Count: 12,281 grid points
    Mean: 2.73 °C-weeks
    Median: 2.32 °C-weeks
    Range: 0.00 - 16.44 °C-weeks
    Std Dev: 2.43 °C-weeks


In [40]:
thermal_clean.to_csv('thermal_caribbean_2016_2018_2020_PROCESSED.csv', index=False)